# Django Case Study III — Git committer dates and LaTeX Table S4

This notebook checks the **exact Git committer timestamps** for the 15 Django checkpoints, using the official [`django/django`](https://github.com/django/django) repository through GitHub's commits API. It generates:

- `django_history_committer_dates.tex` — the updated **Table S4**, retaining all existing rows and adding *Commit date* and *Source component* columns, with `\revision{}` marking new material.
- `django_history_committer_dates.csv` — a machine-readable provenance table including complete timestamp, UTC.

**Run in Google Colab:** Runtime → Run all. No API key, Google Drive connection, or extra packages are required. GitHub applies a rate limit to unauthenticated API requests; the notebook reports a clear error if this is reached rather than inventing dates.

In [ ]:
from pathlib import Path
import csv
import json
import urllib.request
import urllib.error
from datetime import datetime, timezone

# Dataset from existing Supplementary Material, Table S4. Order and role are unchanged.
CHECKPOINTS = [
    (r"Ticket \#30289", "e245046b", "Vulnerable", "Admin permission checks"),
    ("", "8335d592", "Corrected", ""),
    ("", "0027ca76", "Post-fix 1", ""),
    ("", "f9f09a8a", "Post-fix 2", ""),
    ("", "7191beec", "Post-fix 3", ""),
    ("", "3d465f77", "Post-fix 4", ""),
    ("", "3e01aab5", "Post-fix 5", ""),
    ("", "93daed25", "Post-fix 6", ""),
    ("", "7cbcf2e2", "Post-fix 7", ""),
    ("CVE-2019-12308", "98c0fe19", "Vulnerable", "Admin URL rendering"),
    ("", "deeba6d9", "Corrected", ""),
    ("CVE-2021-31542", "8de4ca74", "Vulnerable", "Uploaded-file and storage paths"),
    ("", "0b79eb36", "Corrected", ""),
    ("CVE-2021-33203", "f66ae7a2", "Vulnerable", "Admin documentation path handling"),
    ("", "46572de2", "Corrected", ""),
]
assert len(CHECKPOINTS) == 15
print("Checkpoints loaded:", len(CHECKPOINTS))

## Retrieve and verify commit dates

The value is `commit.committer.date` from GitHub's commit record, **not** `commit.author.date` or the vulnerability disclosure date. GitHub returns an ISO timestamp in UTC. The table uses the `YYYY-MM-DD` portion of that timestamp. The notebook checks that the returned full SHA begins with the expected revision prefix.

In [ ]:
def fetch_commit(sha):
    url = f"https://api.github.com/repos/django/django/commits/{sha}"
    request = urllib.request.Request(url, headers={
        "Accept": "application/vnd.github+json",
        "User-Agent": "S3DT-TDSC-Django-TableS4/1.0",
        "X-GitHub-Api-Version": "2022-11-28",
    })
    try:
        with urllib.request.urlopen(request, timeout=35) as response:
            commit = json.load(response)
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:350]
        raise RuntimeError(
            f"Could not retrieve {sha} (HTTP {exc.code}). "
            f"Check internet access or GitHub API rate limits. Details: {detail}"
        ) from exc
    except Exception as exc:
        raise RuntimeError(f"Could not retrieve {sha}: {exc}") from exc

    full_sha = commit["sha"]
    if not full_sha.startswith(sha):
        raise RuntimeError(f"Git SHA mismatch: asked for {sha}, got {full_sha}")
    iso_utc = commit["commit"]["committer"]["date"]
    # GitHub's committer date is an ISO 8601 timestamp in UTC.
    return iso_utc, full_sha, commit.get("html_url", url)

rows = []
for issue, short_sha, role, component in CHECKPOINTS:
    commit_time, full_sha, url = fetch_commit(short_sha)
    rows.append({
        "Security history": issue.replace(r"\#", "#"),
        "Git revision": short_sha,
        "Full Git SHA": full_sha,
        "Commit date (UTC)": commit_time[:10],
        "Commit timestamp (UTC)": commit_time,
        "Role": role,
        "Source component": component,
        "GitHub URL": url,
    })
    print(f"{short_sha}  {commit_time[:10]}  {role}")

if len(rows) != len(CHECKPOINTS):
    raise RuntimeError("Not all 15 commits were retrieved; no table should be published.")
print("\nAll 15 Git committer dates retrieved and verified.")

## Generate the revised supplementary table and provenance CSV

The LaTeX preserves the existing Table S4 row order. To compile it, ensure your preamble includes `\usepackage{booktabs,tabularx,array}` and that `\revision{}` is defined in your manuscript.

In [ ]:
OUTPUT_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
TEX_FILE = OUTPUT_DIR / "django_history_committer_dates.tex"
CSV_FILE = OUTPUT_DIR / "django_history_committer_dates.csv"

lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Historical Django checkpoints, Git committer dates, and their roles in the evaluation.}",
    r"\label{tab:django-history}",
    r"\scriptsize",
    r"\begin{tabularx}{\linewidth}{@{}llll>{\raggedright\arraybackslash}X@{}}",
    r"\toprule",
    r"\textbf{Security history} & \textbf{Git revision} & \revision{\textbf{Commit date}} & \textbf{Role} & \revision{\textbf{Source component}} \\",
    r"\midrule",
]
for i, (issue, sha, role, component) in enumerate(CHECKPOINTS):
    if i > 0 and issue:
        lines.append(r"\midrule")
    date = rows[i]["Commit date (UTC)"]
    source = rf"\revision{{{component}}}" if component else ""
    lines.append(rf"{issue} & \texttt{{{sha}}} & \revision{{{date}}} & {role} & {source} " + r"\\")
lines.extend([r"\bottomrule", r"\end{tabularx}", r"\end{table}"])
TEX_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")

with CSV_FILE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Saved:", TEX_FILE)
print("Saved:", CSV_FILE)

try:
    from IPython.display import display, FileLink
    import pandas as pd
    display(pd.DataFrame(rows)[["Security history", "Git revision", "Commit date (UTC)", "Role", "Source component"]])
    display(FileLink(str(TEX_FILE)), FileLink(str(CSV_FILE)))
except ImportError:
    pass

## Optional: download both files to your computer

After the cells above finish successfully, run this last cell in Colab to download the `.tex` and `.csv`. If your browser blocks multiple downloads, use the files sidebar in Colab instead.

In [ ]:
try:
    from google.colab import files
    files.download(str(TEX_FILE))
    files.download(str(CSV_FILE))
except ImportError:
    print("Not running in Colab. Files are saved at:", TEX_FILE, "and", CSV_FILE)